In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from torchvision import models

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

Using: cpu


In [3]:
X = np.load('../data/X.npy')
y = np.load('../data/y.npy')

In [4]:
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)

print(X.shape, y.shape)

torch.Size([28709, 224, 224, 3]) torch.Size([28709])


In [5]:
X = X.permute(0,3,1,2) # (N, H, W, C) -> (N, C, H, W)
print(X.shape)

torch.Size([28709, 3, 224, 224])


In [6]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=32)

In [7]:
class EmotionCnn(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(3,32,3),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32,64,3),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.AdaptiveAvgPool2d((1,1))

        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64,7)
        )

    def forward(self,x):
        x = self.conv(x)
        x = self.fc(x)
        return x

In [8]:
print(X.shape)

torch.Size([28709, 3, 224, 224])


In [10]:
model = models.resnet18(pretrained=True)

model.fc = nn.Linear(model.fc.in_features, 7)

model = model.to(device)

for param in model.parameters():
    param.requires_grad = False

for param in model.fc.parameters():
    param.requires_grad = True

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=0.001)

for epoch in range(10):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for i, (batch_X, batch_y) in enumerate(train_loader):
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)
        preds = model(batch_X)
        loss = loss_fn(preds, batch_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()*batch_X.size(0)

        _ , predicted = torch.max(preds, 1)
        correct += (predicted == batch_y).sum().item()
        total += batch_y.size(0)

        if i % 100 == 0:
            print(f"Epoch {epoch+1} | Batch {i}/{len(train_loader)} | Loss: {loss.item():.4f}")

    train_acc = correct/total
    
    #validation
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)
            preds = model(batch_X)
            _ , predicted = torch.max(preds, 1)

            correct += (predicted == batch_y).sum().item()
            total += batch_y.size(0)

    val_acc = correct/total
    
    avg_loss = total_loss / len(train_loader.dataset)
    print(f"Epoch {epoch+1}, Loss: {avg_loss:.3f}, Train Acc: {train_acc:.3f} Val Acc: {val_acc:.3f}")

Epoch 1 | Batch 0/718 | Loss: 2.1189
Epoch 1 | Batch 100/718 | Loss: 1.6540
Epoch 1 | Batch 200/718 | Loss: 1.6111
Epoch 1 | Batch 300/718 | Loss: 1.5466
Epoch 1 | Batch 400/718 | Loss: 1.6761
Epoch 1 | Batch 500/718 | Loss: 1.6829
Epoch 1 | Batch 600/718 | Loss: 1.6225
Epoch 1 | Batch 700/718 | Loss: 1.6046
Epoch 1, Loss: 1.630, Train Acc: 0.360 Val Acc: 0.412
Epoch 2 | Batch 0/718 | Loss: 1.5276
Epoch 2 | Batch 100/718 | Loss: 1.3932
Epoch 2 | Batch 200/718 | Loss: 1.5487
Epoch 2 | Batch 300/718 | Loss: 1.3898
Epoch 2 | Batch 400/718 | Loss: 1.3941
Epoch 2 | Batch 500/718 | Loss: 1.3669
Epoch 2 | Batch 600/718 | Loss: 1.4086
Epoch 2 | Batch 700/718 | Loss: 1.4275
Epoch 2, Loss: 1.511, Train Acc: 0.415 Val Acc: 0.421
Epoch 3 | Batch 0/718 | Loss: 1.6137
Epoch 3 | Batch 100/718 | Loss: 1.4151
Epoch 3 | Batch 200/718 | Loss: 1.2913
Epoch 3 | Batch 300/718 | Loss: 1.3959
Epoch 3 | Batch 400/718 | Loss: 1.2778
Epoch 3 | Batch 500/718 | Loss: 1.6947
Epoch 3 | Batch 600/718 | Loss: 1.6211
E

In [11]:
torch.save(model.state_dict(), 'emotion_model_v2.pth')